# 03 — Evaluation: Fine-Tuned Mistral 7B vs GPT-4o-mini
Compare fine-tuned model against GPT-4o-mini (Azure) baseline on 500 eval examples.

In [ ]:
!pip install -q transformers peft bitsandbytes datasets openai rapidfuzz wandb python-dotenv

In [ ]:
import os
import sys
import json
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv()
from src.data.format import load_jsonl
from src.data.schema import Invoice
from src.evaluation.evaluate import load_finetuned_model, run_finetuned_inference, aggregate_metrics, generate_report
from src.evaluation.baseline import run_baseline
import wandb
wandb.init(project="invoice-extraction-finetune", job_type="evaluation")

## Load Eval Data

In [ ]:
eval_data = load_jsonl("../data/eval.jsonl")
print(f"Loaded {len(eval_data)} eval examples")
gold_invoices = []
for ex in eval_data:
    gold_invoices.append(Invoice.model_validate_json(ex["output"]))

## Run Fine-Tuned Model

In [ ]:
model, tokenizer = load_finetuned_model(
    base_model="mistralai/Mistral-7B-v0.3",
    adapter_path="../outputs/final_adapter",
)
ft_predictions = run_finetuned_inference(model, tokenizer, eval_data)
ft_metrics = aggregate_metrics(ft_predictions, gold_invoices)
print(f"Fine-tuned overall accuracy: {ft_metrics['overall_accuracy']:.1%}")
print(f"JSON parse rate: {ft_metrics['json_parse_success_rate']:.1%}")

## Run GPT-4o-mini Baseline

In [ ]:
baseline_predictions = run_baseline(eval_data)
baseline_metrics = aggregate_metrics(baseline_predictions, gold_invoices)
print(f"Baseline overall accuracy: {baseline_metrics['overall_accuracy']:.1%}")
print(f"JSON parse rate: {baseline_metrics['json_parse_success_rate']:.1%}")

## Comparison Report

In [ ]:
report = generate_report(ft_metrics, baseline_metrics)
print(report)

os.makedirs("../reports", exist_ok=True)
with open("../reports/evaluation_report.md", "w") as f:
    f.write(report)

wandb.log({
    "ft_overall_accuracy": ft_metrics["overall_accuracy"],
    "baseline_overall_accuracy": baseline_metrics["overall_accuracy"],
    "ft_parse_rate": ft_metrics["json_parse_success_rate"],
    "baseline_parse_rate": baseline_metrics["json_parse_success_rate"],
    "improvement_pct": (
        (ft_metrics["overall_accuracy"] - baseline_metrics["overall_accuracy"])
        / baseline_metrics["overall_accuracy"] * 100
    ),
})

for field_name in ft_metrics["per_field"]:
    wandb.log({
        f"ft_{field_name}": ft_metrics["per_field"].get(field_name, 0),
        f"baseline_{field_name}": baseline_metrics["per_field"].get(field_name, 0),
    })

wandb.finish()
print("\nReport saved to reports/evaluation_report.md")
print("Metrics logged to W&B")